# Expérience — Ablation des features

**Objectif :** Déterminer quelles features sont utiles, inutiles ou nuisibles au modèle.

Trois séries de tests :
1. **Ablation** des features géographiques/linguistiques (`num_languages`, `num_countries`, `is_english`)
2. **Réintroduction** des features supprimées lors du nettoyage (`budget`, `revenue`, `adult`)
3. **Ablation fine** sur la meilleure config trouvée en (1) — une feature retirée à la fois, puis par paires

**Référence :** Baseline 10 features → Accuracy **66%** | Macro F1 **0.66**

> Basé sur `notebooks/baseline.ipynb` — setup identique (même `RANDOM_STATE`, même genres, même undersampling).

## Setup — Reproduction du pipeline baseline

In [1]:
import pandas as pd
import numpy as np
from itertools import combinations

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report

RANDOM_STATE = 42
SELECTED_GENRES = ['Animation', 'Horror', 'Drama']

In [2]:
# Chargement
df = pd.read_csv("hf://datasets/HenryWaltson/TMDB-IMDB-Movies-Dataset/TMDB  IMDB Movies Dataset.csv")

# Nettoyage
df = df.drop_duplicates()
df = df.drop(columns=['backdrop_path', 'keywords', 'homepage', 'tconst', 'overview', 'poster_path', 'tagline'])
df = df[df['release_date'].notna()]

# Feature engineering
total_votes = df['vote_count'] + df['numVotes']
df['rating']      = (df['vote_average'] * df['vote_count'] + df['averageRating'] * df['numVotes']) / total_votes
df['total_votes'] = total_votes
df = df.drop(columns=['vote_count', 'numVotes', 'vote_average', 'averageRating'])

df['is_english']    = (df['original_language'] == 'en').astype(int)
df['cast_count']    = df['cast'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)
df['release_month'] = pd.to_datetime(df['release_date']).dt.month
df['release_year']  = pd.to_datetime(df['release_date']).dt.year
df['num_languages'] = df['spoken_languages'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)
df['num_countries'] = df['production_countries'].fillna('').apply(lambda x: len(x.split(',')) if x else 0)
df['adult_int']     = df['adult'].astype(int)

# Label
df_clean = df[df['genres'].notna()].copy()
df_clean['genre'] = df_clean['genres'].str.split(',').str[0].str.strip()

# Sélection genres + undersampling
df_final = df_clean[df_clean['genre'].isin(SELECTED_GENRES)].copy()
cap = df_final['genre'].value_counts().min()
df_balanced = df_final.groupby('genre', group_keys=False).sample(n=cap, random_state=RANDOM_STATE)

print(f"Dataset prêt : {len(df_balanced):,} films ({cap:,} par genre)")
print(df_balanced['genre'].value_counts())

/Users/adam/Desktop/ECE/ING4/S8/Apprentissage et Estimation Bayesienne/Projet/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Dataset prêt : 58,203 films (19,401 par genre)
genre
Animation    19401
Drama        19401
Horror       19401
Name: count, dtype: int64


## Fonction utilitaire

In [3]:
CONTINUOUS_BASE = ['rating', 'total_votes', 'popularity', 'runtime']

def run_experiment(name, features, continuous_override=None):
    """Entraîne et évalue un pipeline GaussianNB sur un jeu de features donné."""
    cont = continuous_override if continuous_override else [f for f in CONTINUOUS_BASE if f in features]
    pas  = [f for f in features if f not in cont]

    transforms = []
    if cont: transforms.append(('scale', RobustScaler(), cont))
    if pas:  transforms.append(('pass', 'passthrough', pas))

    X_tr, X_te, y_tr, y_te = train_test_split(
        df_balanced[features], df_balanced['genre'],
        test_size=0.2, stratify=df_balanced['genre'], random_state=RANDOM_STATE
    )
    pipe = Pipeline([('preprocessor', ColumnTransformer(transforms)), ('model', GaussianNB())])
    pipe.fit(X_tr, y_tr)

    report = classification_report(y_te, pipe.predict(X_te), output_dict=True)
    return {
        'name':     name,
        'acc':      report['accuracy'],
        'macro_f1': report['macro avg']['f1-score'],
        'anim_f1':  report['Animation']['f1-score'],
        'drama_f1': report['Drama']['f1-score'],
        'horror_f1':report['Horror']['f1-score'],
        'features': features,
    }

def print_results(results, sort_by='macro_f1'):
    results = sorted(results, key=lambda r: r[sort_by], reverse=True)
    print(f"{'Expérience':<48} {'Anim':>5} {'Drama':>5} {'Horr':>5} {'Macro':>5} {'Acc':>7}")
    print('-' * 78)
    for r in results:
        marker = ' ◀ baseline' if r['name'].startswith('Baseline') else ''
        print(f"{r['name']:<48} {r['anim_f1']:.2f}  {r['drama_f1']:.2f}  {r['horror_f1']:.2f}  {r['macro_f1']:.2f}  {r['acc']:.2%}{marker}")

---
## Expérience 1 — Ablation des features géographiques / linguistiques

On teste l'impact de `num_languages`, `num_countries` et `is_english`, seules et combinées.

In [4]:
ALL_FEATURES = ['rating', 'total_votes', 'popularity', 'runtime',
                'is_english', 'cast_count', 'release_month', 'release_year',
                'num_languages', 'num_countries']

ablation_tests = {
    'Baseline (toutes, 10 features)':                        [],
    'Sans num_languages':                                    ['num_languages'],
    'Sans is_english':                                       ['is_english'],
    'Sans num_countries':                                    ['num_countries'],
    'Sans num_languages + is_english':                       ['num_languages', 'is_english'],
    'Sans num_languages + num_countries':                    ['num_languages', 'num_countries'],
    'Sans is_english + num_countries':                       ['is_english', 'num_countries'],
    'Sans les 3 (num_languages, is_english, num_countries)': ['num_languages', 'is_english', 'num_countries'],
}

results_exp1 = []
for name, to_remove in ablation_tests.items():
    feat = [f for f in ALL_FEATURES if f not in to_remove]
    results_exp1.append(run_experiment(name, feat))

print_results(results_exp1)

Expérience                                        Anim Drama  Horr Macro     Acc
------------------------------------------------------------------------------
Sans num_languages + num_countries               0.73  0.66  0.62  0.67  67.14%
Sans num_countries                               0.73  0.65  0.62  0.67  66.91%
Sans num_languages                               0.72  0.65  0.62  0.67  66.80%
Sans les 3 (num_languages, is_english, num_countries) 0.73  0.65  0.61  0.66  66.68%
Sans num_languages + is_english                  0.72  0.65  0.62  0.66  66.44%
Sans is_english + num_countries                  0.72  0.65  0.61  0.66  66.41%
Baseline (toutes, 10 features)                   0.72  0.64  0.62  0.66  66.31% ◀ baseline
Sans is_english                                  0.72  0.64  0.61  0.66  65.98%


---
## Expérience 2 — Réintroduction de features supprimées (`budget`, `revenue`, `adult`)

Ces colonnes existent dans le dataset mais n'ont pas été utilisées (beaucoup de valeurs nulles pour budget/revenue).  
On teste si leur ajout apporte quelque chose, en partant de la meilleure config de l'expérience 1 (8 features, sans les 3 géo/linguistiques).

In [5]:
BASE_8 = ['rating', 'total_votes', 'popularity', 'runtime',
          'is_english', 'cast_count', 'release_month', 'release_year']

reintro_tests = {
    'Baseline 8 features':        [],
    '+ budget':                   ['budget'],
    '+ revenue':                  ['revenue'],
    '+ adult':                    ['adult_int'],
    '+ budget + revenue':         ['budget', 'revenue'],
    '+ budget + adult':           ['budget', 'adult_int'],
    '+ revenue + adult':          ['revenue', 'adult_int'],
    '+ budget + revenue + adult': ['budget', 'revenue', 'adult_int'],
}

results_exp2 = []
for name, to_add in reintro_tests.items():
    feat = BASE_8 + to_add
    # budget et revenue sont continues, adult_int est binaire
    cont_extra = [f for f in to_add if f in ['budget', 'revenue']]
    cont = CONTINUOUS_BASE + cont_extra
    results_exp2.append(run_experiment(name, feat, continuous_override=cont))

print_results(results_exp2)

Expérience                                        Anim Drama  Horr Macro     Acc
------------------------------------------------------------------------------
Baseline 8 features                              0.73  0.66  0.62  0.67  67.14% ◀ baseline
+ adult                                          0.72  0.63  0.62  0.66  65.90%
+ revenue                                        0.03  0.50  0.02  0.18  33.59%
+ revenue + adult                                0.03  0.50  0.02  0.18  33.59%
+ budget + revenue                               0.03  0.02  0.49  0.18  33.12%
+ budget + revenue + adult                       0.03  0.02  0.49  0.18  33.12%
+ budget                                         0.03  0.00  0.49  0.18  33.04%
+ budget + adult                                 0.03  0.00  0.49  0.18  33.04%


---
## Expérience 3 — Ablation fine sur la meilleure config

On part de la meilleure config identifiée (à mettre à jour selon résultats exp 1 & 2) et on retire chaque feature une à une, puis par paires.

In [7]:
# Meilleure config identifiée en exp 1 : sans num_languages et num_countries
BEST_FEATURES = ['rating', 'total_votes', 'popularity', 'runtime',
                 'is_english', 'cast_count', 'release_month', 'release_year']

ablation_fine = {'Baseline meilleure config': []}
for f in BEST_FEATURES:
    ablation_fine[f'Sans {f}'] = [f]
for f1, f2 in combinations(BEST_FEATURES, 2):
    ablation_fine[f'Sans {f1} + {f2}'] = [f1, f2]

results_exp3 = []
for name, to_remove in ablation_fine.items():
    feat = [f for f in BEST_FEATURES if f not in to_remove]
    if len(feat) < 2:
        continue
    results_exp3.append(run_experiment(name, feat))

print_results(results_exp3)

Expérience                                        Anim Drama  Horr Macro     Acc
------------------------------------------------------------------------------
Sans runtime                                     0.73  0.66  0.63  0.67  67.42%
Sans runtime + release_month                     0.73  0.66  0.63  0.67  67.37%
Baseline meilleure config                        0.73  0.66  0.62  0.67  67.14% ◀ baseline
Sans release_month + release_year                0.73  0.67  0.62  0.67  67.30%
Sans release_month                               0.73  0.66  0.62  0.67  67.13%
Sans total_votes + runtime                       0.73  0.67  0.61  0.67  67.17%
Sans total_votes                                 0.73  0.67  0.61  0.67  67.26%
Sans release_year                                0.72  0.67  0.61  0.67  67.22%
Sans total_votes + release_month                 0.73  0.67  0.61  0.67  67.20%
Sans runtime + is_english                        0.73  0.65  0.61  0.67  66.68%
Sans total_votes + popularity

---
## Conclusions

### Expérience 1 — Features géo/linguistiques
- `num_languages` et `num_countries` **dégradent** le modèle → à retirer
- `is_english` est légèrement utile → à garder
- Meilleure config exp 1 : **8 features** (sans `num_languages` + `num_countries`) → Acc **67.14%** | Macro F1 **0.67**

### Expérience 2 — Réintroduction budget / revenue / adult
- `budget` et `revenue` **font s'effondrer le modèle à ~33%** — environ 75% des films ont ces valeurs à 0 (données manquantes encodées en 0), ce qui crée une distribution aberrante pour la gaussienne → **à ne jamais réintroduire**
- `adult` dégrade légèrement (−1%) → inutile

### Expérience 3 — Ablation fine
Classement par importance décroissante (impact de la suppression sur le Macro F1) :
1. `cast_count` — **critique** (−3% seul, jusqu'à −4.5% en combo)
2. `rating` — **critique** (−3% seul)
3. `popularity` — **important** (−2% seul)
4. `total_votes`, `release_year`, `release_month` — utiles mais secondaires
5. `is_english` — contribution faible (~stable sans)
6. `runtime` — **légèrement négatif**, le retirer améliore de +0.28%

### Config finale recommandée — 7 features

```python
features = ['rating', 'total_votes', 'popularity',
            'is_english', 'cast_count', 'release_month', 'release_year']
```

**Résultat attendu : Acc ~67.4% | Macro F1 ~0.67** (vs 66.3% / 0.66 baseline 10 features)

> Propager cette config dans `main.ipynb` : mettre à jour `features`, `continuous_features` et `passthrough_features`.